# 🏆 Porra del Mundial 2026 · Actualizar Ranking

**Instrucciones:**
1. Pulsa **Entorno de ejecución → Ejecutar todo** (`Ctrl+F9`)
2. Acepta el permiso de Google Drive cuando te lo pida
3. El ranking se calcula y se publica en Netlify automáticamente ✅

---

In [ ]:
# ── Paso 1: Conectar Google Drive ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive conectado')

In [ ]:
# ── Paso 2: Configuración ──────────────────────────────────────────
#
# Google Drive
RUTA_CARPETA = 'Mi unidad/Porra Mundial'   # <-- cambia si hace falta
#
# Netlify
#   · Token  → app.netlify.com/user/applications → "New access token"
#   · Site ID → déjalo vacío la primera vez; el script lo crea y te lo muestra
#               En siguientes ejecuciones pega el ID aquí para ir más rápido
#
NETLIFY_TOKEN   = 'PEGA_AQUI_TU_TOKEN'   # <-- pon tu token (no lo compartas)
NETLIFY_SITE_ID = ''                      # <-- vacío la 1ª vez; luego pega el ID

# ──────────────────────────────────────────────────────────────────
import os
RAIZ = f'/content/drive/{RUTA_CARPETA}'

if not os.path.isdir(RAIZ):
    raise FileNotFoundError(f'No se encuentra: {RAIZ} — revisa RUTA_CARPETA')

os.chdir(RAIZ)
print(f'📁 {RAIZ}')
!ls

In [ ]:
# ── Paso 3: Instalar dependencias ─────────────────────────────────
!pip install --quiet openpyxl requests
print('✅ Dependencias listas')

In [ ]:
# ── Paso 4: Ver pronósticos recibidos ─────────────────────────────
import glob
archivos = [
    f for f in sorted(glob.glob('pronosticos/*.xlsx'))
    if not os.path.basename(f).startswith('~$')
]
print(f'📋 Pronósticos ({len(archivos)}):')
for f in archivos:
    print(f'   · {os.path.basename(f)}')
if not archivos:
    print('⚠️  Sin archivos en pronosticos/ todavía.')

In [ ]:
# ── Paso 5: Calcular ranking y generar dashboard ───────────────────
import subprocess, sys

def run(script):
    r = subprocess.run([sys.executable, script], cwd=RAIZ, capture_output=True, text=True)
    if r.stdout: print(r.stdout)
    if r.returncode != 0:
        print('❌ Error:'); print(r.stderr)
        raise RuntimeError(f'Falló {script}')

print('⚙️  Generando dataset de partidos...')
run('scripts/build_matches.py')
print('⚙️  Calculando puntos...')
run('scripts/scorer.py')
print('✅ Dashboard generado')

In [ ]:
# ── Paso 6: Publicar en Netlify ────────────────────────────────────
import requests, zipfile, io, random, string

if not NETLIFY_TOKEN or NETLIFY_TOKEN == 'PEGA_AQUI_TU_TOKEN':
    print('⚠️  NETLIFY_TOKEN vacío — salto la publicación.')
else:
    HEADERS = {'Authorization': f'Bearer {NETLIFY_TOKEN}'}

    # Empaquetar dashboard.html como index.html
    html_path = os.path.join(RAIZ, 'dashboard', 'dashboard.html')
    with open(html_path, 'rb') as f:
        html_bytes = f.read()

    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.writestr('index.html', html_bytes)
    buf.seek(0)

    site_id = NETLIFY_SITE_ID.strip()

    # Crear sitio si no hay ID
    if not site_id:
        print('🆕 Creando sitio en Netlify...')
        sufijo = ''.join(random.choices(string.ascii_lowercase + string.digits, k=5))
        r = requests.post(
            'https://api.netlify.com/api/v1/sites',
            headers={**HEADERS, 'Content-Type': 'application/json'},
            json={'name': f'porra-mundial-{sufijo}'}
        )
        site = r.json()
        site_id = site['id']
        print(f'   Sitio: {site["ssl_url"]}')
        print(f'   ⚠️  Guarda este ID en NETLIFY_SITE_ID para la próxima vez:')
        print(f'   NETLIFY_SITE_ID = "{site_id}"')

    # Deploy
    print('🚀 Publicando...')
    r = requests.post(
        f'https://api.netlify.com/api/v1/sites/{site_id}/deploys',
        headers={**HEADERS, 'Content-Type': 'application/zip'},
        data=buf.getvalue()
    )

    if r.status_code in (200, 201):
        deploy = r.json()
        url = deploy.get('ssl_url') or deploy.get('url', '(pendiente)')
        print(f'')
        print(f'✅ ¡Publicado!')
        print(f'   🔗 {url}')
        print(f'   Comparte ese link con los participantes.')
    else:
        print(f'❌ Error Netlify {r.status_code}: {r.text}')

In [ ]:
# ── Paso 7 (opcional): Vista previa del ranking ────────────────────
import json

with open('dashboard/data.json', encoding='utf-8') as f:
    data = json.load(f)

print(f"🏆 {data.get('torneo')} · {data.get('actualizado')}")
print()
print(f"{'Pos':>3}  {'Participante':<20} {'Total':>6}  {'Plenos':>6}  {'Tend.':>6}")
print('─' * 48)
for i, p in enumerate(data.get('ranking', []), 1):
    print(f"{i:>3}  {p['alias']:<20} {p['total']:>6}  {p['pleno']:>6}  {p['tendencia']:>6}")

---

## 📌 Instrucciones para los participantes

1. **Descarga** la plantilla `plantillas/plantilla_porra.xlsx` desde la carpeta compartida
2. **Rellena** tus apuestas y guarda el archivo como `porra_TUNOMBRE.xlsx`
3. **Súbelo** a la carpeta `pronosticos/` de la carpeta compartida

¡Listo! El administrador actualizará el ranking ejecutando este notebook.